# 01b · K vecinos más cercanos: la geometría de la distancia

**Módulo 4 · Sesión 9** — KNN y SVM

## Objetivos

`02-knn-y-svm.md` describe KNN en una frase: buscar los $k$ puntos más cercanos y votar.
Este notebook **dibuja** esa frase, con dos variables y pocos estudiantes para que todo se
vea:

1. Clasificar un punto nuevo como **inflar un círculo** a su alrededor hasta que contenga
   $k$ vecinos, y por qué la respuesta cambia con $k$.
2. KNN a mano en cinco líneas, validado contra `KNeighborsClassifier`.
3. Con $k = 1$, el plano se reparte en **celdas de Voronoi** y la frontera está hecha de
   trozos de **mediatrices**.
4. $k$ como la perilla sesgo-varianza: de la frontera que rodea cada punto a la que ignora
   los datos, y por qué las probabilidades de KNN vienen en escalones.
5. Por qué KNN **exige** escalar: un círculo en unidades crudas es una elipse aplastada
   que solo mira una variable.
6. La maldición de la dimensionalidad, medida: columnas de ruido que hunden a KNN y, 🔵
   opcional, por qué en muchas dimensiones "el más cercano" deja de significar algo.

Usa el `rendimiento-estudiantes.csv` de los módulos 1 y 3 y del notebook 01, ahora para
predecir `aprobo` con `promedio_anterior` y `horas_estudio_semana`.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-4-clasificacion-ensambles/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle, Ellipse
from scipy.spatial import Voronoi, voronoi_plot_2d
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Veinte estudiantes en el plano

Tomamos **20 estudiantes** al azar de los 400: suficientes para que haya vecinos, pocos
para poder seguir cada uno con el ojo. Las dos variables están en escalas muy distintas
—el promedio va de 2 a 5, las horas de estudio de 0 a 30—, así que por ahora las
trabajamos **estandarizadas** ($z = (x - \bar x)/s$): una unidad en cada eje es una
desviación estándar, y un círculo es un círculo. La sección 6 muestra qué pasa si no.

In [ ]:
datos = pd.read_csv("../datos/rendimiento-estudiantes.csv")
VARIABLES = ["promedio_anterior", "horas_estudio_semana"]

muestra = datos.sample(n=20, random_state=SEMILLA)
X_crudo = muestra[VARIABLES].to_numpy()
y = muestra["aprobo"].to_numpy()

media, desv = X_crudo.mean(axis=0), X_crudo.std(axis=0)
X = (X_crudo - media) / desv

print(f"Aprobaron {y.sum()} de {len(y)}")
muestra[["id_estudiante", *VARIABLES, "aprobo"]].reset_index(drop=True)

In [ ]:
COLORES = {0: "tab:red", 1: "tab:blue"}
FONDO = ListedColormap(["#f4c7c3", "#c6dbef"])
LIM = (-2.6, 2.6)


def preparar_eje(eje, titulo=""):
    eje.set_xlim(*LIM)
    eje.set_ylim(*LIM)
    eje.set_aspect("equal")
    eje.set_xlabel("promedio_anterior (estandarizado)")
    eje.set_ylabel("horas_estudio (estandarizado)")
    eje.set_title(titulo)


def dibujar_puntos(eje, X, y, s=45):
    for clase, nombre in [(0, "no aprobó"), (1, "aprobó")]:
        eje.scatter(*X[y == clase].T, color=COLORES[clase], s=s, edgecolor="black",
                    linewidth=0.5, label=nombre, zorder=3)


fig, eje = plt.subplots(figsize=(5.5, 5.5))
dibujar_puntos(eje, X, y)
eje.legend(loc="upper left")
preparar_eje(eje, "20 estudiantes")
plt.show()

Los que aprobaron (azul) tienden a estar arriba a la derecha —más promedio, más horas—,
pero hay mezcla en el centro. No hay ninguna recta trazada: KNN no la va a necesitar.

## 2. Un estudiante nuevo e inflar un círculo

Llega un estudiante que no está en los datos. KNN no tiene parámetros que aplicarle: lo
único que hace es medir su **distancia euclidiana** a cada uno de los 20,

$$
d(\mathbf{x}, \mathbf{x}_i) = \sqrt{(x_1 - x_{i1})^2 + (x_2 - x_{i2})^2},
$$

ordenarlas y votar entre los $k$ primeros. Geométricamente es **inflar un círculo**
centrado en el punto nuevo hasta que encierre $k$ estudiantes.

In [ ]:
NUEVO = np.array([-0.8, -0.25])  # en unidades estandarizadas

distancias = np.sqrt(np.sum((X - NUEVO) ** 2, axis=1))
orden = np.argsort(distancias)

tabla = pd.DataFrame(
    {"distancia": distancias[orden].round(3), "aprobo": y[orden]},
    index=pd.Index(range(1, 21), name="vecino"),
)
tabla.head(7)

In [ ]:
K_MOSTRAR = [1, 3, 7]

fig, ejes = plt.subplots(1, 3, figsize=(15, 5.2))
for eje, k in zip(ejes, K_MOSTRAR):
    radio = distancias[orden[k - 1]]
    votos = y[orden[:k]]
    clase = int(votos.mean() > 0.5)
    eje.add_patch(Circle(NUEVO, radio, facecolor=COLORES[clase], alpha=0.12,
                         edgecolor=COLORES[clase], linewidth=1.5, zorder=1))
    for i in orden[:k]:
        eje.plot(*np.c_[NUEVO, X[i]], color="gray", linewidth=0.8, zorder=2)
    dibujar_puntos(eje, X, y)
    eje.scatter(*NUEVO, marker="*", s=260, color=COLORES[clase], edgecolor="black", zorder=4)
    preparar_eje(eje, f"k = {k}: {votos.sum()} de {k} aprobaron → "
                      f"{'aprueba' if clase else 'no aprueba'}")
plt.tight_layout()
plt.show()

Con $k = 1$ el vecino más cercano no aprobó, y KNN predice que el estudiante nuevo
tampoco. Con $k = 3$ el círculo crece, entran dos aprobados y la predicción se invierte.
Con $k = 7$ crece otra vez, alcanza a tres no aprobados más del lado izquierdo, y la
predicción vuelve a "no aprueba". **No hay una respuesta "correcta" independiente de $k$**: $k$ es la escala a la
que se mira el vecindario. Un círculo pequeño es muy sensible a quién está justo al lado;
uno grande promedia, pero mezcla estudiantes cada vez menos parecidos.

La probabilidad que reporta KNN es la fracción de votos: $\hat p = 0/1$, $2/3$ o $3/7$
según el caso. Con $k$ vecinos solo puede tomar $k + 1$ valores (sección 5).

## 3. KNN a mano

El algoritmo cabe en cinco líneas: distancias de cada punto nuevo a todos los de
entrenamiento, los índices de las $k$ menores, y la fracción de unos entre sus etiquetas.

In [ ]:
def knn_proba(X_entren, y_entren, X_nuevos, k):
    """P(clase 1) por voto de los k vecinos más cercanos (distancia euclidiana)."""
    # (m, 1, p) - (1, n, p) se difunde a (m, n, p): todas las diferencias de una vez
    d = np.sqrt(np.sum((X_nuevos[:, None, :] - X_entren[None, :, :]) ** 2, axis=-1))
    vecinos = np.argsort(d, axis=1)[:, :k]
    return y_entren[vecinos].mean(axis=1)


# Validación sobre una malla de 200 x 200 puntos que cubre todo el plano
g1, g2 = np.meshgrid(np.linspace(*LIM, 200), np.linspace(*LIM, 200))
malla = np.c_[g1.ravel(), g2.ravel()]

for k in [1, 3, 5, 9]:
    a_mano = knn_proba(X, y, malla, k)
    sk = KNeighborsClassifier(n_neighbors=k).fit(X, y).predict_proba(malla)[:, 1]
    print(f"k = {k}: máxima diferencia con scikit-learn = {np.abs(a_mano - sk).max():.1e}")

Coinciden en los 40 000 puntos de la malla. Fíjese en lo que **no** hay en
`knn_proba`: ningún `fit`, ningún parámetro aprendido, ningún bucle de optimización. El
"modelo" son `X_entren` y `y_entren`, y todo el trabajo ocurre al predecir — $m \times n$
distancias para $m$ puntos nuevos. Es el costo $O(np)$ por predicción de
`02-knn-y-svm.md`.

## 4. Con $k = 1$: celdas de Voronoi

Pintemos el plano entero según lo que predice KNN con $k = 1$. Cada estudiante se queda
con la región de puntos que están **más cerca de él que de cualquier otro**: su **celda de
Voronoi**. El borde entre dos celdas vecinas es el conjunto de puntos equidistantes de
ambos estudiantes, es decir, un trozo de la **mediatriz** del segmento que los une.

In [ ]:
def pintar_regiones(eje, proba, niveles=None, cmap=FONDO):
    z = proba.reshape(g1.shape)
    if niveles is None:  # solo la clase predicha
        eje.contourf(g1, g2, (z > 0.5).astype(int), levels=[-0.5, 0.5, 1.5], cmap=cmap,
                     zorder=0)
    else:
        eje.contourf(g1, g2, z, levels=niveles, cmap=cmap, zorder=0)


vor = Voronoi(X)
fig, ejes = plt.subplots(1, 2, figsize=(12, 5.8))

pintar_regiones(ejes[0], knn_proba(X, y, malla, 1))
voronoi_plot_2d(vor, ax=ejes[0], show_points=False, show_vertices=False,
                line_colors="gray", line_width=0.8, line_alpha=0.7)
dibujar_puntos(ejes[0], X, y)
preparar_eje(ejes[0], "k = 1: el plano partido en celdas de Voronoi")

# Solo las aristas que separan estudiantes de clases distintas: esa es la frontera
pintar_regiones(ejes[1], knn_proba(X, y, malla, 1))
for (i, j), aristas in zip(vor.ridge_points, vor.ridge_vertices):
    if y[i] != y[j] and -1 not in aristas:
        ejes[1].plot(*vor.vertices[aristas].T, color="black", linewidth=2)
        medio = (X[i] + X[j]) / 2
        ejes[1].plot(*np.c_[X[i], X[j]], color="gray", linestyle=":", linewidth=1)
        ejes[1].plot(*medio, "s", color="gray", markersize=3)
dibujar_puntos(ejes[1], X, y)
preparar_eje(ejes[1], "La frontera: mediatrices entre clases distintas")
plt.tight_layout()
plt.show()

A la izquierda, las 20 celdas: cada punto del plano hereda la clase del estudiante que
"posee" su celda. A la derecha, solo las aristas que separan a un aprobado de un no
aprobado (en negro), con el segmento que une a esa pareja en punteado: la arista corta el
segmento por la mitad y en ángulo recto. La frontera de KNN con $k=1$ es una
**poligonal hecha de mediatrices** (las aristas que se van al infinito no se dibujan,
pero siguen la misma regla).

Tres consecuencias que se leen en la figura:

- **Acierto perfecto en entrenamiento.** Cada estudiante cae en su propia celda: el error
  de entrenamiento con $k=1$ es cero siempre (salvo filas duplicadas con etiquetas
  distintas). Por eso la precisión de entrenamiento no dice nada sobre KNN.
- **Islas.** En esta muestra las dos clases están bastante separadas, pero un no
  aprobado rodeado de aprobados se quedaría con su celda roja en medio del azul: si es
  ruido, KNN lo memoriza. Con los 400 estudiantes hay muchas islas así.
- **Duplicados.** Una fila de prueba idéntica a una de entrenamiento cae exactamente
  sobre su gemelo, a distancia cero. Es el mecanismo detrás de la fuga que
  `02-clasificacion-aplicado.ipynb` mide en Wine Quality.

## 5. $k$ como perilla sesgo-varianza

Al subir $k$ cada punto del plano consulta un vecindario más grande: las islas
desaparecen y la frontera se suaviza. En el extremo $k = n$ todos los puntos consultan a
**los mismos** 20 estudiantes y KNN predice siempre la clase mayoritaria.

In [ ]:
K_PANEL = [1, 3, 7, 20]

fig, ejes = plt.subplots(1, 4, figsize=(19, 5))
for eje, k in zip(ejes, K_PANEL):
    proba = knn_proba(X, y, malla, k)
    pintar_regiones(eje, proba)
    eje.contour(g1, g2, proba.reshape(g1.shape), levels=[0.5], colors="black",
                linewidths=1.2)
    dibujar_puntos(eje, X, y, s=35)
    acierto = np.mean((knn_proba(X, y, X, k) > 0.5) == y)
    preparar_eje(eje, f"k = {k}  (acierto en entrenamiento {acierto:.2f})")
plt.tight_layout()
plt.show()

> Con $k$ par puede haber empates en la votación (2 contra 2). Aquí un empate,
> $\hat p = 0.5$, cuenta como "no aprueba"; `scikit-learn` desempata eligiendo la primera
> clase, que es la misma regla. Por eso se suele preferir $k$ impar en problemas binarios.

### Las probabilidades vienen en escalones

Con $k$ vecinos, $\hat p$ solo puede valer $0, 1/k, 2/k, \dots, 1$: el mapa de
probabilidades es una **escalera** de a lo sumo $k + 1$ peldaños, no una rampa como la
sigmoide de la logística (notebook 01).

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.8))
for eje, k in zip(ejes, [1, 5, 11]):
    proba = knn_proba(X, y, malla, k)
    niveles = np.linspace(-0.5 / k, 1 + 0.5 / k, k + 2)  # un color por cada valor posible
    pintar_regiones(eje, proba, niveles=niveles, cmap="RdBu")
    dibujar_puntos(eje, X, y, s=30)
    preparar_eje(eje, f"k = {k}: p̂ toma {len(np.unique(proba))} de {k + 1} valores posibles")
plt.tight_layout()
plt.show()

Con $k = 11$ no aparecen los 12 peldaños: con solo 6 no aprobados entre los 20, ningún
vecindario de 11 tiene más de 6 rojos, y $\hat p$ nunca baja de $5/11$: solo quedan 7
peldaños. Para una curva ROC o PR esa granularidad importa:
con $k=5$ solo hay 6 umbrales distintos donde cortar, y la curva tiene a lo sumo 6
vértices.

### La curva en U, con los 400 estudiantes

Con 20 puntos se ve la forma de la frontera; para medir el error hacen falta más datos.
Sobre los 400 estudiantes, con validación cruzada estratificada de 10 pliegues y el
escalado **dentro** del `Pipeline` (para que cada pliegue se estandarice solo con sus
filas de entrenamiento):

In [ ]:
X_todos = datos[VARIABLES].to_numpy()
y_todos = datos["aprobo"].to_numpy()
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEMILLA)

valores_k = [1, 3, 5, 7, 11, 15, 21, 31, 51, 75, 101, 151, 201, 301]
filas = []
for k in valores_k:
    modelo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    acc_cv = cross_val_score(modelo, X_todos, y_todos, cv=cv, scoring="accuracy")
    acc_entren = modelo.fit(X_todos, y_todos).score(X_todos, y_todos)
    filas.append({"k": k, "entrenamiento": acc_entren, "validación": acc_cv.mean(),
                  "error estándar": acc_cv.std(ddof=1) / np.sqrt(len(acc_cv))})
curva_k = pd.DataFrame(filas)

fig, eje = plt.subplots(figsize=(8, 4.5))
eje.plot(curva_k["k"], curva_k["entrenamiento"], "o-", label="entrenamiento")
eje.errorbar(curva_k["k"], curva_k["validación"], yerr=curva_k["error estándar"], fmt="o-",
             capsize=3, label="validación cruzada (10 pliegues)")
eje.axhline(y_todos.mean(), color="gray", linestyle="--",
            label=f"predecir siempre 'aprueba' ({y_todos.mean():.2f})")
eje.set_xscale("log")
eje.set_xlabel("k  (escala log; a la izquierda, modelo más complejo)")
eje.set_ylabel("accuracy")
eje.legend()
plt.tight_layout()
plt.show()

curva_k.round(3)

Es la curva en U de `05-sesgo-varianza-validacion.md` (módulo 3), con el eje de
complejidad **al revés**: $k$ pequeño es el modelo flexible. Con $k=1$ el acierto de
entrenamiento es 1.0 y el de validación (0.775) queda por debajo de cualquier $k$ entre
3 y 101; con $k$ cercano a $n$ los
dos caen hacia el 0.74 de predecir siempre la clase mayoritaria. Entre ambos hay una
meseta ancha: cualquier $k$ entre 11 y 75 da entre 0.81 y 0.83, diferencias del tamaño
del error estándar (≈ 0.01).

## 6. Sin escalar, el círculo es una elipse

Todo lo anterior usó variables estandarizadas. Veamos qué hace KNN con las variables
**crudas**. La desviación estándar de las horas de estudio es varias veces la del
promedio —unas 7 en esta muestra de 20, casi 11 en los 400—, así que una diferencia
típica en horas pesa ese múltiplo más en la distancia:

$$
d^2 = (\Delta\,\text{promedio})^2 + (\Delta\,\text{horas})^2
= s_p^2\,(\Delta z_p)^2 + s_h^2\,(\Delta z_h)^2
$$

El conjunto de puntos a distancia cruda $r$ ya no es un círculo en el plano estandarizado,
sino una **elipse** con semiejes $r/s_p$ (a lo largo del promedio) y $r/s_h$ (a lo largo
de las horas): $s_h/s_p$ veces más ancha que alta. El "vecindario" se convierte en una franja
horizontal que solo exige parecerse en horas.

In [ ]:
desv_400 = datos[VARIABLES].std(ddof=0).to_numpy()
print(f"Desviación estándar en la muestra — promedio: {desv[0]:.3f}   horas: {desv[1]:.3f}   "
      f"razón: {desv[1] / desv[0]:.1f}")
print(f"Desviación estándar en los 400   — promedio: {desv_400[0]:.3f}   horas: {desv_400[1]:.3f}   "
      f"razón: {desv_400[1] / desv_400[0]:.1f}")

k = 5
d_estand = np.sqrt(np.sum((X - NUEVO) ** 2, axis=1))
nuevo_crudo = NUEVO * desv + media
d_crudo = np.sqrt(np.sum((X_crudo - nuevo_crudo) ** 2, axis=1))

fig, ejes = plt.subplots(1, 2, figsize=(12, 5.8))
for eje, d, titulo in [(ejes[0], d_estand, "Distancia con variables estandarizadas"),
                       (ejes[1], d_crudo, "Distancia con variables crudas")]:
    vecinos = np.argsort(d)[:k]
    radio = d[vecinos[-1]]
    if d is d_estand:
        forma = Circle(NUEVO, radio)
    else:  # elipse de radio crudo, dibujada en coordenadas estandarizadas
        forma = Ellipse(NUEVO, 2 * radio / desv[0], 2 * radio / desv[1])
    forma.set(facecolor="gold", alpha=0.25, edgecolor="goldenrod", linewidth=1.5, zorder=1)
    eje.add_patch(forma)
    dibujar_puntos(eje, X, y)
    eje.scatter(*X[vecinos].T, s=180, facecolor="none", edgecolor="black", linewidth=1.5,
                zorder=4, label=f"los {k} vecinos")
    eje.scatter(*NUEVO, marker="*", s=260, color="gold", edgecolor="black", zorder=5)
    eje.legend(loc="upper left")
    preparar_eje(eje, titulo)
plt.tight_layout()
plt.show()

Mismo estudiante nuevo, mismo $k$, vecinos distintos. Sin escalar, KNN elige como
"parecidos" a estudiantes con promedios muy distintos solo porque estudian un número de
horas similar. La figura está en coordenadas estandarizadas para que la comparación sea
justa: en las unidades crudas la franja sería un círculo, pero el eje del promedio
ocuparía una décima parte del de horas.

El efecto sobre la frontera, con los 400 estudiantes y $k = 51$:

In [ ]:
def frontera_400(eje, modelo, titulo):
    modelo.fit(X_todos, y_todos)
    p1, p2 = np.meshgrid(np.linspace(2.0, 5.0, 250), np.linspace(0, 30, 250))
    proba = modelo.predict_proba(np.c_[p1.ravel(), p2.ravel()])[:, 1].reshape(p1.shape)
    eje.contourf(p1, p2, (proba > 0.5).astype(int), levels=[-0.5, 0.5, 1.5], cmap=FONDO)
    eje.contour(p1, p2, proba, levels=[0.5], colors="black", linewidths=1.2)
    for clase in (0, 1):
        eje.scatter(*X_todos[y_todos == clase].T, color=COLORES[clase], s=8, alpha=0.6)
    eje.set_xlabel("promedio_anterior")
    eje.set_ylabel("horas_estudio_semana")
    eje.set_title(titulo)


fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
frontera_400(ejes[0], make_pipeline(StandardScaler(), KNeighborsClassifier(51)),
             "k = 51, escalado")
frontera_400(ejes[1], KNeighborsClassifier(51), "k = 51, sin escalar")
plt.tight_layout()
plt.show()

filas = []
for k in [5, 15, 51, 101]:
    filas.append({
        "k": k,
        "escalado": cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)),
                                    X_todos, y_todos, cv=cv).mean(),
        "sin escalar": cross_val_score(KNeighborsClassifier(k), X_todos, y_todos, cv=cv).mean(),
    })
pd.DataFrame(filas).set_index("k").round(3)

Con las variables crudas la frontera es casi **horizontal**: KNN decide por las horas de
estudio y apenas mira el promedio, que es justamente la variable más predictiva. El
costo se ve en la validación cruzada, más cuanto mayor es $k$ —la franja crece y mezcla
estudiantes con promedios cada vez más distintos—: con $k = 51$ se pierden 4 puntos, y
con $k = 101$ KNN sin escalar ya no le gana a predecir siempre "aprueba" (0.74).

Escalar no es un detalle de preprocesamiento: **define** qué significa "parecido". Y no
hay una escala neutral. Estandarizar le da a cada variable el mismo peso en la distancia;
si una variable importa más que otra para el problema, KNN no lo sabe — por eso le va mal
con variables irrelevantes, que es lo que sigue.

## 7. La maldición de la dimensionalidad

KNN supone que "cerca" significa "parecido en lo que importa". Cada variable que no
importa añade ruido a la distancia. Medimos qué pasa al agregar a los dos predictores
reales columnas de **ruido gaussiano puro**, ya estandarizadas como las demás.

In [ ]:
filas = []
for extra in [0, 2, 5, 10, 20, 50, 100]:
    X_ruido = np.hstack([X_todos, rng.normal(size=(len(X_todos), extra))])
    for k in [15, 51]:
        modelo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
        acc = cross_val_score(modelo, X_ruido, y_todos, cv=cv)
        filas.append({"columnas de ruido": extra, "k": k, "accuracy": acc.mean()})
ruido = pd.DataFrame(filas).pivot(index="columnas de ruido", columns="k", values="accuracy")

fig, eje = plt.subplots(figsize=(7, 4))
for k in ruido.columns:
    eje.plot(ruido.index, ruido[k], "o-", label=f"k = {k}")
eje.axhline(y_todos.mean(), color="gray", linestyle="--", label="clase mayoritaria")
eje.set_xscale("symlog", linthresh=1)
eje.set_xlabel("columnas de ruido añadidas a las 2 reales")
eje.set_ylabel("accuracy (validación cruzada)")
eje.legend()
plt.tight_layout()
plt.show()

ruido.round(3)

Con solo dos columnas de ruido, KNN ($k=15$) baja de 0.82 a 0.78 y pierde la mitad de su
ventaja sobre la clase mayoritaria; con 50 llega exactamente al 0.74 de predecir siempre
"aprueba". Las dos variables útiles quedan sepultadas en una distancia que suma muchas
diferencias irrelevantes, y KNN no tiene forma de saber cuáles ignorar. Los modelos que
eligen **una** variable en cada pregunta resisten mucho mejor: en el notebook 03
(sección 6), bagging de árboles con 18 columnas de ruido sigue en 0.89. Por eso con KNN
la selección de variables (`05-seleccion-y-pipelines.md`, módulo 2) no es opcional.

### 🔵 Opcional: en muchas dimensiones todos están lejos

Hay un segundo efecto, que no depende de que las variables sean ruido. Tomemos 1000
puntos uniformes en el cubo $[0, 1]^p$ y midamos, desde un punto cualquiera, la distancia
al más cercano y al más lejano.

In [ ]:
filas = []
fig, ejes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=False)
for p, eje in zip([2, 10, 100], ejes):
    puntos = rng.uniform(size=(1000, p))
    d = np.sqrt(np.sum((puntos[1:] - puntos[0]) ** 2, axis=1))
    eje.hist(d, bins=40, color="tab:gray")
    eje.set_title(f"p = {p}: distancias desde un punto")
    eje.set_xlabel("distancia")
for p in [2, 5, 10, 50, 100, 500, 1000]:
    puntos = rng.uniform(size=(1000, p))
    d = np.sqrt(np.sum((puntos[1:] - puntos[0]) ** 2, axis=1))
    filas.append({"p": p, "más cercano": d.min(), "más lejano": d.max(),
                  "lejano / cercano": d.max() / d.min(),
                  "lado del cubo con el 1 % de los datos": 0.01 ** (1 / p)})
plt.tight_layout()
plt.show()

pd.DataFrame(filas).set_index("p").round(3)

Al crecer $p$, el histograma de distancias se vuelve un pico estrecho: el vecino más
cercano y el más lejano están a distancias **casi iguales** (la razón entre ellas tiende
a 1). "El más cercano" deja de distinguir a nadie.

La última columna lo dice de otra forma. Para que un vecindario cúbico contenga el 1 % de
datos uniformes, su lado debe ser $0.01^{1/p}$: un 10 % del rango en 2 dimensiones, pero
el 63 % en 10 y el 95 % en 100. Con muchas variables, los $k$ vecinos más cercanos de un
punto **no son locales**: el vecindario ocupa casi todo el rango de cada variable.

Los datos reales rara vez llenan el cubo de manera uniforme —suelen vivir cerca de
superficies de baja dimensión, y por eso KNN sigue funcionando en muchos problemas—,
pero la intuición se mantiene: KNN funciona con pocas variables relevantes y muchos
datos; mal al revés.

## Resumen

| Lo que se vio | Su forma geométrica | Conecta con |
|---|---|---|
| Predicción de KNN | Inflar un círculo hasta encerrar $k$ vecinos y votar | `02-knn-y-svm.md`, sección 1 |
| "Entrenar" KNN | Guardar los datos; todo el costo está en predecir | Costo $O(np)$ por predicción |
| $k = 1$ | Celdas de Voronoi; frontera hecha de mediatrices | Acierto 1.0 en entrenamiento; fuga por duplicados (notebook 02) |
| $k$ | Tamaño del vecindario: islas con $k$ pequeño, clase mayoritaria con $k = n$ | Curva en U, `05-sesgo-varianza-validacion.md` (módulo 3) |
| $\hat p$ de KNN | Escalera de $k + 1$ peldaños, no una rampa | Curvas ROC y PR (notebook 02) |
| Sin escalar | El círculo se vuelve una elipse; decide la variable de mayor varianza | `StandardScaler` dentro del `Pipeline` |
| Variables de ruido | Diluyen la distancia hasta la clase mayoritaria | Selección de variables (módulo 2); árboles (notebook 03) |
| 🔵 Alta dimensión | Todas las distancias se parecen; el vecindario deja de ser local | Reducción de dimensionalidad (módulo 5) |

El notebook 01c cambia la pregunta: en vez de mirar a los vecinos de cada punto nuevo,
busca **una sola frontera** —la más ancha posible— y se olvida de casi todos los datos.